<a href="https://colab.research.google.com/github/sofiarubini02/textual-entailment-nlp/blob/main/textual_entailment_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROGETTO MACHINE LEARNING: Textual Entailment (NLI)**


CORSO: FILOSOFIA E INTELLIGENZA ARTIFICIALE, 2024/2025





In quanto studenti di Filosofia e IA, il tema del Natural Language Processing  risulta particolarmente accattivante: strumento principale del processo di interazione uomo-macchina, è un campo di studi multidisciplinare che racchiude logica, semantica, linguistica e informatica.  
Considerato un problema IA-completo, in quanto richiede una estesa conoscenza del mondo, è stato interessante e stimolante per noi poterlo trattare con mano più ingegneristica.


# **INSTALLAZIONE E IMPORTAZIONE LIBRERIE/PACCHETTI**

**PIP INSTALL:**

In [ ]:
!pip install --quiet torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu118
!pip install tqdm datasets --quiet
!pip install --quiet pytorch_lightning

NB: L'errore che vediamo è un warning di [incompatibilità](https://stackoverflow.com/questions/72672196/error-pips-dependency-resolver-does-not-currently-take-into-account-all-the-pa) tra le versioni di gcsfs (libreria che serve a leggere e scrivere file su Google Cloud Storage) che richiede la versione 2025.3.2 di fsspec (libreria su cui gcsfs si appoggia per gestire filesystem virtuali.), e la versione fsspec 2024.12.0 già installata su Colab.

Abbiamo provato a forzare l'installazione di versioni meno recenti in modo da soddisfare tutte le dipendenze ed evitare warning ed errori per rendere il notebook più pulito, ma dopo aver approdato questa modifica ci siamo accorti che le prestazioni di tutti i modelli calavano di qualche punteggio nelle metriche di valutazione.

Perciò, dato che nel progetto non useremo Google Cloud o dataset HuggingFace più avanzati, abbiamo deciso di ignorare questo warning che non ci creerà nessun problema, salvaguardando il best score dei modelli.

**LIBRERIE BASE PER ANALISI DATI E VISUALIZZAZIONE:**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**DATASET HUGGING FACE:**

In [ ]:
from datasets import load_dataset

**SCIKIT-LEARN: ML, METRCHE E SPLIT:**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

**UTILITY PER NLP:**

In [ ]:
import string
from collections import Counter

**PROGRESS BAR:**

In [ ]:
from tqdm import tqdm
from tqdm.notebook import tqdm as notebook_tqdm


**PYTORCH:**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


NB: Notiamo un warning preventivo legato alla [compatibilità](https://github.com/stardist/stardist/issues/297) tra NumPy 1.x (usato da alcuni moduli Python) e NumPy 2.x (la nostra versione attuale). La soluzione sarebbe quella di fare il downgrade di NumPy alla verisone 1.24.4, ma così facendo avremo altri problemi di incompatibilità tra i pip install. Dato che il codice funziona correttamente, possiamo ignorare il warning."


**PYTORCH LIGHTNING:**



In [ ]:
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, TQDMProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

**IMPOSTAZIONI GRAFICHE PER I NOTEBOOK:**

Utilizzeremo due funzionalità che rendono il notebook più interpretabile:
- [%matplotlib inline](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-matplotlib): rende i grafici visualizzabili direttamente nel notebook  
- [pd.set_option]( https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html): mostra più testo per ogni riga

In [ ]:
%matplotlib inline
sns.set(style="whitegrid")
pd.set_option('display.max_colwidth', 200)

**DOWNLOAD (AUTOMATICO) DEL DATASET**

Installiamo la libreria 'datasets' di Hugging Face (piattaforma open source che permette di scaricare automaticamente il dataset) e procediamo al caricamento del dataset, suddiviso in training, validation e test set.

Infine, visualizziamo le prime righe del training set.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("snli")

dataset["train"].to_pandas().head()


Visualizziamo il dataset:

*   La premise è la frase di partenza
*   La hypothesis è la frase da verificare (rispetto alla premise)
*   La label è l'etichetta che rappresenta la relazione tra le due frasi: 0 (implicazione), 1 (neutrale), 2 (contraddizione), -1 (valore mancante)

Il significato delle label è scritto nella [documentazione ufficiale](https://huggingface.co/datasets/snli)



**CONVERSIONE DEL DATASET SNLI IN DATAFRAME PANDAS:**

Convertiamo i tre split del dataset SNLI (`train`, `validation`, `test`) in oggetti Pandas DataFrame per facilitarne l'esplorazione e il pre-processin: Pandas infatti è una libreria che ci consente di visualizzare in forma tabellare i dati, filtrandoli e manipolandoli più facilmente.

Infine visualizziamo le prime righe del training (per verificare che non sia cambiato nulla).  


In [ ]:

df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()
df_test = dataset["test"].to_pandas()

df_train.head()

# **PRE-PROCESSING**

**MAPPATURA DELLE ETICHETTE**

Definiamo un dizionario di mappatura tra etichette numeriche (0,1,2) e le loro rispettive classi testuali ("entailment", "neutral", "contradiction").

Aggiungiamo una nuova colonna `label_text` nel DataFrame del training set.


In [ ]:
label_map = {
    0: "entailment",
    1: "neutral",
    2: "contradiction"
}

df_train["label_text"] = df_train["label"].map(label_map)

df_train[["premise", "hypothesis", "label", "label_text"]].head()

**ANALISI DEI VALORI NULLI**

Dopo aver fatto la label map, verifichiamo la presenza di valori nulli (che nel nostro caso non hanno una label text): usiamo il comando isnull che restituisce True se ci sono valori nulli, e applichiamo una somma per avere il numero totale.


In [ ]:
df_train.isnull().sum()

**DATA CLEANING**

Dopo aver accertato che esistono dei valori nulli/mancanti nel dataset, filtriamo solo le righe che hanno una label valida (0,1,2).

Ricreiamo la colonna `label_text` nella mappa delle etichette utilizzando il trainingset aggiornato.

Infine facciamo una verifica finale.


In [ ]:
df_train = df_train[df_train["label"].isin([0, 1, 2])]

df_train["label_text"] = df_train["label"].map(label_map)

df_train["label"].unique(), df_train["label_text"].isnull().sum()

Il warning ci avvisa che stiamo modificando una copia, e non il dataframe originale.  

**ANALISI DELLA DISTRIBUZIONE DELLE CLASSI NEL TRAINING SET**

Analizziamo la distribuzione delle classi nel training set per farci un'idea sul loro bilanciamento: contiamo le labels testuali per ogni classe e mostriamo il risultato in un grafico.


In [ ]:
print(df_train["label_text"].value_counts())


sns.countplot(x="label_text", data=df_train)
plt.title("Distribuzione delle classi nel training set")
plt.xlabel("Classe")
plt.ylabel("Numero di esempi")
plt.show()

Il training set è ben equilibrato, senza classi dominanti. Ottima base per un training efficace.

**ANALISI STATISTICA PRE-TOKENIZZAZIONE**

Per una più completa analisi esplorativa dei dati, calcoliamo la lunghezza (in parole) di premise e hypothesis: avremo così la lunghezza massima delle frasi, utile per scoprire eventuali outliers e per impostare un numero massimo di token nella fase di addestramento.




In [ ]:
df_train["premise_len"] = df_train["premise"].apply(lambda x: len(str(x).split()))
df_train["hypothesis_len"] = df_train["hypothesis"].apply(lambda x: len(str(x).split()))

df_train[["premise", "premise_len", "hypothesis", "hypothesis_len"]].head()


Facciamo una analisi statistica sulla lunghezza delle frasi, calcolando valori come media, deviazione standard, minimo e massimo.

Queste informazioni aiutano a determinare la lunghezza massima da usare nella tokenizzazione e nel padding delle sequenze testuali (evitare tokenizzazione e padding eccessivi).


In [ ]:
df_train[["premise_len", "hypothesis_len"]].describe()


Questo riepilogo statistico ci da le seguenti informazioni:
* count= quante frasi hai analizzato (dovrebbe coincidere con il numero di righe)
* mean= lunghezza media delle frasi (in parole)
* std= scarto quadratico medio (variazione rispetto alla media)
* min= frase più corta (in parole)
* 25%= il 25% delle frasi ha lunghezza uguale o inferiore a questo valore
* 50%= mediana: metà delle frasi è più corta, metà più lunga
* 75%= il 75% delle frasi è più corta o uguale a questo valore
* max= frase più lunga nel dataset

I nostri risultati sono buoni:
* Le premise sono in media più lunghe (≈13 parole)
* Le hypothesis sono più brevi (≈7 parole)
La distribuzione è compatta: il 75% delle frasi ha premise ≤ 16 parole e hypothesis ≤ 9 parole. Questo significa che il dataset è piuttosto compatto, senza frasi lunghissime

I valori estremi sono rari: sono poche le frasi che arrivano al massimo (78 e 56 parole); questo significa che potrebbero essere troncate per risparmiare spazio, senza perdere molta informazione.

**ISTOGRAMMI DELLE LUNGHEZZE:**

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df_train["premise_len"], bins=30, kde=True)
plt.title("Distribuzione lunghezza delle 'premise'")
plt.xlabel("Numero di parole")
plt.ylabel("Frequenza")
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(df_train["hypothesis_len"], bins=30, kde=True, color='orange')
plt.title("Distribuzione lunghezza delle 'hypothesis'")
plt.xlabel("Numero di parole")
plt.ylabel("Frequenza")
plt.show()

**BOXPLOT COMPARATIVO:**

Il grafico permette di visualizzare la distribuzione, la mediana, i quartili e gli eventuali outlier. È utile per valutare differenze nella complessità delle due tipologie di frasi e decidere un `max_length` appropriato per la tokenizzazione.


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_train[["premise_len", "hypothesis_len"]])
plt.title("Boxplot lunghezze: Premise vs Hypothesis")
plt.ylabel("Numero di parole")
plt.show()

# **BASELINE (Logistic Regression)**

Il  task è una classificazione testuale multilabel: come punto di partenza utilizziamo un modello di Logistic Regression adatto a classificazioni multiclasse (con ScikitLearn), leggero e veloce da implementare anche su grandi dataset (come il nostro) e efficace nel comprendere sfumature linguistiche (grazie alla trasformazione di parole in vettori).  

**CONCATENAZIONE**


Fondamentale è la concatenazione tra premise e hypothesis in un'unica stringa, in quanto la Logistic Regression lavora su un’unica sequenza di testo trasformata in numeri (e non su coppie separate di frasi come nel nostro dataset).

In mezzo alle due frasi aggiungiamo il token speciale SEP che permette al modello di distinguere pattern testuali prima e dopo il SEP (anche se non è veramente "capace" di comprendere la sintassi).

Questa struttura in input è adatta per essere vettorizzata con `TfidfVectorizer`che trasforma l'intera frase concatenata in una rappresentazione numerica (se lasciamo le frasi separate, avremmo 2 vettorizzazioni diverse da combinare manualmente).


In [ ]:
df_train["text"] = df_train["premise"] + " [SEP] " + df_train["hypothesis"]

df_train[["text", "label_text"]].head()

**VETTORIZZAZIONE DEL TESTO CON TF-IDF E CREAZIONE DELLE ETICHETTE**

Sia TF-IDF che Hashing sono valide tecniche di vettorizzazione del testo, ma siccome stiamo usando una Logistic Regression per un task di entailment optiamo per la vettorizzazione con TF-IDF (Term Frequency – Inverse Document Frequency), una tecnica per trasformare un testo in un vettore numerico interpretabile, dove ogni numero rappresenta quanto una parola è importante in un documento, rispetto a tutti gli altri documenti (con hashing invece non potremmo pesare l'importanza delle parole).  



Utilizziamo `TfidfVectorizer` per trasformare le frasi combinate (`premise` + `hypothesis`) in vettori numerici.
 Specifichiamo un massimo di 10.000 features per iniziare (dato che il dataset contiene più di 500000 frasi) e analizziamo singoli pezzi di frase (unigrammi e bigrammi, ovverso singole parole e coppie di parole)

Trasformiamo la colonna text (che contiene la concatenazione delle frasi) in una matrice sparsa di tipo (numero_frasi, numero_feature).

Le etichette target (`y`) vengono estratte come vettori numerici per l’addestramento (0,1,2).


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))


X = vectorizer.fit_transform(df_train["text"])

y = df_train["label"]


**VERIFICA DELLA DIMENSIONE DELLA MATRICE TF-IDF**

Prima di procedere, verifichiamo la dimensione della matrice TF-IDF. Il numero di righe corrisponde al numero di esempi nel training set, mentre il numero di colonne rappresenta il numero di feature (unigrammi o bigrammi) estratte.



In [ ]:
X.shape

Otteniamo 549.367 frasi (una per ogni riga del  dataset). Ogni frase è rappresentata da un vettore con 10.000 dimensioni (una per ogni unigramma/bigramma scelto tra le top più frequenti).

Visualizziamo le prime 20 features:

In [ ]:
vectorizer.get_feature_names_out()[:20]


**TRAINING E VALIDATION SET**

Dopo aver trasformato il testo in vettori TF-IDF (X) e aver ottenuto le etichette (y), dividiamo il dataset in training e validation set.

Suddividiamo il dataset vettorizzato in training set (80%) e validation set (20%) utilizzando `train_test_split': si tratta di una scelta standard all'interno della [documentazione ScikitLearn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), con stratificazione basata sulle etichette per garantire che la distribuzione delle classi venga mantenuta in entrambi i sottogruppi.

Usiamo anche random_state=42 per fissare la "casualità" dello split affinchè la divisione rimanga invariata.

Infine verifichiamo la dimensione dei due set risultanti.


In [ ]:

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


X_train.shape, X_val.shape


**ADDESTRAMENTO DEL MODELLO LR**

Iniziamo l'addestramento!

Inizializziamo il modello:
* Impostiamo il `max_iter=1000` per aumentare la possibilità di convergenza del modello, evitando l'overfitting.
* Il solver è l’algoritmo di ottimizzazione che cerca di trovare i pesi (coefficenti) migliori per il modello, cioè quelli che minimizzano l’errore tra le previsioni e i valori reali. Scegliamo il solver `'saga'` è adatto per dataset ampi, informazioni sparse (la matrice TF-IDF è sparsa) e supporta sia L1 che L2 regularization.
* `n_jobs=-1`indica quanti core utilizzare durante l'addestramento: stiamo dicendo di usarli tutti, aumentando la velocità di calcolo.

Addestriamo sui vettori TF-IDF (X) e sulle etichette (y).


In [ ]:
model = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1)

model.fit(X_train, y_train)

**VALUTAZIONE DEL MODELLO LR**

Finito l'addestramento, valutiamo la performance del modello sul validation set.
* Generiamo le predizioni (`y_pred`) sui dati di validazione.
* Stampiamo il `classification_report` che include precision, recall e F1-score per ciascuna classe.
* Visualizziamo anche la `confusion_matrix`, utile per trovare gli errori tra classi.


In [ ]:
y_pred = model.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=["entailment", "neutral", "contradiction"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Il modello riesce a distinguere le classi in modo discreto ma non eccellente.
L’accuracy è bilanciata (59%), ma si nota una difficoltà nel classificare la classe "neutral", che presenta il valore F1 più basso.
Dalla confusion matrix emergono confusioni tra "entailment" e "neutral", e tra "contradiction" e "neutral", indicando una sovrapposizione semantica in molte frasi.


!DESCLAIMER: I valori che analizziamo nella cella di valutaizone del modello, per TUTTI i modelli che seguiranno in questo progetto, sono da intendersi APPORSSIMATI, e NON esatti. I modelli neurali non sono completamente deterministici e dunque ad ogni run i valori possono cambiare leggermente. Abbiamo comunque deciso di analizzare i valori delle metriche, per ogni modello addestrato, in modo da spiegare il comportamento e la qualità dell'addestramento.

**OTTIMIZZAZIONE**

L'accuracy non è bassa per essere un modello basico, ma possiamo provare ad aumentare il numero di features per vedere se migliora.
Costruiamo dunque una nuova vettorizzazione TF-IDF con 30k features e utilizziamo n-grammi fino a 3, per aiutare il modello a cogliere strutture sintattiche più complesse.

Proseguiamo dunque a reiterare il codice di addestramento precedente con questa nuova vettorizzazione.



In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 3))

X = vectorizer.fit_transform(df_train["text"])

y = df_train["label"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_val.shape

In [ ]:
modelO = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1)

modelO.fit(X_train, y_train)

**VALUTAZIONE DEL MODELLO OTTIMIZZATO**

In [ ]:
y_pred = modelO.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=["entailment", "neutral", "contradiction"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))


Non ci sono miglioramenti!

Si potrebbe pensare di ottimizzare il modello bilanciando le classi (class_weight='balanced'), ma come avevamo sottolineato durante l'esplorazione del dataset iniziale, le classi sono già bilanciate.

Concludiamo che il modello sembra essere al limite di quello che può fare con feature-based methods (TF-IDF + Logistic Regression).

--------------------------------------------------------

# **RECURRENT NEURAL NETWORK: LSTM**


Dato il task di NLP, la scelta dell'utilizzo di una rete LSTM (Long Short-Term Memory) sembra essere adatta: questo tipo di RNN infatti è capace di memorizzare informazioni nel tempo e imparare le dipendenze sequenziali (il significato di una frase, soprattutto nel nostro contesto, dipende dall’ordine delle parole).


**VERIFICA DEL DEVICE:**


Lavorando su Google Colab, data la dimensione del dataset e la complessità dei modelli che andremo a utilizzare, prestando attenzione al tempo di esecuzione, la scelta di utilizzare la GPU (Graphic Processing Unit) risulta la più efficace grazie alla sua capacità di lavorare in parallelo su grandi quantità di dati.
Quindi è indispensabile avere installato Pytorch che ci permette di lavorare sulla GPU grazie a CUDA, compilatore di NVIDIA che permette di usare la scheda grafica per calcoli generali, in particolare usiamo la versione compatibile con la GPU NVIDIA di Colab: CUDA 11.8.

Verifichiamo quindi che la versione sia stata correttamente installata, ovvero se Pytorch riesce a vedere la GPU via CUDA.


In [ ]:

print("GPU disponibile:", torch.cuda.is_available())
print("Nome GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna GPU")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


**PRE-PROCESSING:**


Dato che LSTM non lavora con le stringhe di parole vogliamo convertire il testo in una lista pulita di parole.

Iniziamo la fase di standardizzazione (elimina le differenze di carattere), seguita dalla tokenizzazione (che fa uno split sugli spazi).

Applichiamo il pre-processing ad una frase di esempio, per verificare che il processo sia andato a buon fine:

In [ ]:
def preprocess(sentence):
    return sentence.lower().split()


example = "La FILOSOFIA sarà di supporto alle Nuove Tecnologie."
print(preprocess(example))

Ci rendiamo conto della presenza del punto "."

Risolviamo utilizzando la funzione di rimozione della punteggiatura [(string.punctuation) ]( https://docs.python.org/3/library/string.html#string.punctuation)

In [ ]:
def tokenize(text):
    text = text.lower()
    tokens = text.split()
    tokens = [token.strip(string.punctuation) for token in tokens]
    return tokens

print(tokenize(example))


**COSTRUZIONE DEL VOCABOLARIO SUI DATI TOKENIZZATI:**



Dato che LSTM lavora con liste di parole, risulta quindi indispensabile creare un vocabolario, partendo dalla tokenizzazione delle frasi `premise` e `hypothesis`:
*  associamo un indice ad ogni token
* aggiungiamo due nuove colonne con le liste di token
* uniamo tutti i token di premesse e ipotesi in un'unica lista (all_tokens)
* contiamo la frequenza di ogni parola
* costruiamo il dizionario con l'aggiunta di due token special (<PAD> per il padding e <UNK> per le parole sconosciute)
* associamo ogni parola ad un ID univoco nel vocabolario

In [ ]:
tqdm.pandas(desc="Tokenizing")

df_train["premise_tokens"] = df_train["premise"].progress_apply(tokenize)
df_train["hypothesis_tokens"] = df_train["hypothesis"].progress_apply(tokenize)


all_tokens = []
for prem, hypo in tqdm(zip(df_train["premise_tokens"], df_train["hypothesis_tokens"]), total=len(df_train), desc="Merging tokens"):
    all_tokens.extend(prem)
    all_tokens.extend(hypo)


token_freq = Counter(all_tokens)

vocab = {"<PAD>": 0, "<UNK>": 1}
for token in tqdm(token_freq, desc="Building vocab"):
    vocab[token] = len(vocab)

list(vocab.items())[:10]

**PADDING E TRUNCATION:**


Definiamo la funzione `encode_sentence` che converte una lista di token in una sequenza di indici numerici, usando il vocabolario personalizzato costruito in precedenza.  
- I token non presenti vengono sostituiti con `<UNK>`.
- Le sequenze vengono **padded** o **troncate** a lunghezza fissa (default: 20 token): questo sarà utile per fare batch training.
- Applichiamo la funzione a tutte le righe e otteniamo due colonne con liste di ID numerici pronte per l’Embedding layer.


N.B. Questa cella non è indispensabile per il corretto funzionamento del codice, ma è utile per visualizzare il funzionamento della funzione sui dati.




In [ ]:
def encode_sentence(sentence, vocab, max_length=20):
    tokens = [vocab.get(token, vocab["<UNK>"]) for token in sentence]

    if len(tokens) < max_length:
        tokens += [vocab["<PAD>"]] * (max_length - len(tokens))
    else:
        tokens = tokens[:max_length]

    return tokens

df_train["encoded_premise"] = df_train["premise_tokens"].apply(lambda x: encode_sentence(x, vocab))
df_train["encoded_hypothesis"] = df_train["hypothesis_tokens"].apply(lambda x: encode_sentence(x, vocab))

df_train[["encoded_premise", "encoded_hypothesis"]].head()


**CREAZIONE DEL DATASET PERSONALIZZATO PYTORCH:**

Ricreiamo il dataset, ma in una forma che PyTorch capisce, ovvero in *tensori*.
Definiamo la classe `SNLIDataset`, una sottoclasse di [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html) personalizzata per il dataset SNLI.

Dopo aver importato la classe base per creare un custom dataset PyTorch, definiamo la funzione costruttore (init) che memorizza i dati in input.

Definiamo la funzione len che restituisce la lunghezza totale del dataset (fondamentale per sapere il numero di batch da usare durante l'addestramento, e quante volte iterare il dataset nel training).

Richiamiamo anche la funzione encode_sentence che nella classe Dataset di Pytorch è in modalità [on the fly](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset): le frasi non vengono salvate codificate (in ID numerici) in anticipo, ma trasformate in tempo reale ogni volta che il modello richiede un esempio.

Infine la funzione get_item restituisce un dizionario formato da:
- premise
- hypothesis
- label (entailment, contradiction, neutral)

Le frasi vengono tokenizzate e codificate come sequenze di indici numerici, padded o troncate a lunghezza fissa (`max_len`): questi tensori saranno usati nel training loop.


In [ ]:
class SNLIDataset(Dataset):
    def __init__(self, premises, hypotheses, labels, vocab, max_len=50):
        self.premises = premises
        self.hypotheses = hypotheses
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def encode_sentence(self, tokens):
        encoded = [self.vocab.get(tok, self.vocab["<UNK>"]) for tok in tokens]
        if len(encoded) < self.max_len:
            encoded += [self.vocab["<PAD>"]] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]
        return encoded

    def __getitem__(self, idx):
        prem_tokens = self.premises[idx]
        hypo_tokens = self.hypotheses[idx]
        label = self.labels[idx]

        prem_encoded = self.encode_sentence(prem_tokens)
        hypo_encoded = self.encode_sentence(hypo_tokens)

        return {
            "premise": torch.tensor(prem_encoded, dtype=torch.long),
            "hypothesis": torch.tensor(hypo_encoded, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long)
        }



**CREAZIONE DEI DATALOADER:**

Come prima cosa suddividiamo il dataset SNLI in training set e validation set in modo stratificato, mantenendo cioè la distribuzione delle classi.  
Procediamo alla creazione dei due dataset (train e validation) passando premise_tokens, hypothesis_tokens, e label come liste tokenizzate e utilizzando il vocabolario per codificare ogni frase on the fly in get_item.

Usiamo i due dataset personalizzati per creare i [DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) (una classe che serve a gestire il caricamento dei dati in modo automatico, organizzato e ottimizzato): impostiamo un [batch_size di 128](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html), e la modalità shuffle solo per il training (per il validation vogliamo una valutazione più stabile).

Infine verifichiamo il bilanciamento delle classi con una print:





In [ ]:

BATCH_SIZE = 128

train_df, val_df = train_test_split(
    df_train, test_size=0.2, random_state=42, stratify=df_train["label"]
)

train_dataset = SNLIDataset(
    premises=train_df["premise_tokens"].tolist(),
    hypotheses=train_df["hypothesis_tokens"].tolist(),
    labels=train_df["label"].tolist(),
    vocab=vocab
)

val_dataset = SNLIDataset(
    premises=val_df["premise_tokens"].tolist(),
    hypotheses=val_df["hypothesis_tokens"].tolist(),
    labels=val_df["label"].tolist(),
    vocab=vocab
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=True
)

print("Distribuzione training:", train_df["label"].value_counts(normalize=True))
print("Distribuzione validation:", val_df["label"].value_counts(normalize=True))

Osserviamo che la divisione stratificata risulta corretta: ogni classe è rappresentata in modo proporzionale sia nel training che nella validation. Questo ci assicura che il modello non imparerà a favorire una classe e che le metriche saranno affidabili anche sulla validazione.

**DEFINIZIONE DEL MODELLO: LSTM SEMPLICE**

Ora che abbiamo Dataset, DataLoader, vocabolario, padding e tokenizzazione possiamo iniziare l'allenamento del modello LSTM.

Definiamo la classe del Classifier:
- In init salviamo gli iperparametri, convertiamo gli ID delle parole in vettori densi con l'embedding layer, impostiamo l'unidirezionalità, definiamo il linear layer, e infine la loss function per la classificazione multi-classe (CrossEntropy)
- Nella fase di inferenza (forward) facciamo l'embedding delle frasi, applichiamo l'LSTM a hypotesis e premises, concateniamo i rispettivi ultimi hidden layers e passiamo la concatenazione al linear layer che restituisce un logit per ciascuna classe.
- Nel training step calcoliamo la loss delle previsioni
- Nel validation step valutiamo le prestazioni e calcoliamo l’accuracy batch per batch.
- Scegliamo come ottimizzatore intelligente Adam con un learning rate [standard](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) di 1e-3 (0.001).

NB: Il codice è stato scritto tenendo conto della documentazione ufficiale di [LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html).

In [ ]:
class LSTMClassifier(pl.LightningModule):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, output_dim=3, lr=1e-3, pad_idx=0):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, premise, hypothesis):
        prem_emb = self.embedding(premise)
        hypo_emb = self.embedding(hypothesis)

        _, (prem_hidden, _) = self.lstm(prem_emb)
        _, (hypo_hidden, _) = self.lstm(hypo_emb)

        prem_rep = prem_hidden[-1]
        hypo_rep = hypo_hidden[-1]

        combined = torch.cat([prem_rep, hypo_rep], dim=1)
        output = self.fc(combined)
        return output

    def training_step(self, batch, batch_idx):
        logits = self(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        preds = torch.argmax(logits, dim=1)
        acc = (preds == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


**INIZIALIZZAZIONE DEL MODELLO: LSTM SEMPLICE**

Dopo aver definito la classe del LSTMClassifier, procediamo con l'allenamento del modello.


Inizializziazione: creiamo il modello concreto da addestrare, usando i parametri decisi durante la preparazione.

In [ ]:
model_LSTM = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    output_dim=3,
    lr=1e-3,
    pad_idx=vocab["<PAD>"]
)


**DEFINIZIONE DEL TRAINER:**

Definiamo il Trainer che si occupa di gestire il loop di training, loggare metriche, valutare il modello ad ogni epoca e supportare GPU in automatico.
Aggiungiamo la funzionalità di  EarlyStopping (con patience=5), la barra di avanzamento e la classe [TensorBoardLogger](https://lightning.ai/docs/pytorch/stable/api/pytorch_lightning.loggers.tensorboard.html) che serve per salvare log e visualizzare grafici interattivi.



In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="lstm_model"
)


progress_bar = TQDMProgressBar(refresh_rate=10)


trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    callbacks=[early_stop, progress_bar],
    logger=logger
)



Per ogni epoca:
- Viene mostrata la progress bar (grazie a TQDMProgressBar)
- Il Logger salva le metriche nella cartella lightning_logs
- L'Early stopping monitora la metrica scelta (val_loss)

Dopo aver definito la classe LSTMClassifier, preparato train_loader e val_loader, creato il Trainer con early stopping, barra di avanzamento e logger, iniziamo l'addestramento!



**ADDESTRAMENTO:**

In [ ]:
trainer.fit(model_LSTM, train_loader, val_loader)


Dopo ogni training, aggiungeremo il seguente codice per visualizzare graficamente i logger:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/lstm_model/


**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**

Concludiamo quindi con la valutazione del modello.
Importiamo da sklearn la confusion matrix e il classification_report che ci permetterà di visualizzare le seguenti metriche di valutazione: Precision, Accuracy, Recall, F1Score.


Definiamo, all'interno della funzione evaluate_model, la model_eval, la quale attiva la modalità inference (no dropout, no gradient) e [disattiviamo il calcolo del gradiente](https://pytorch.org/docs/stable/generated/torch.no_grad.html).



In [ ]:

def evaluate_model(model, dataloader):
    model = model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione in corso"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())


    return all_labels, all_preds


y_true, y_pred = evaluate_model(model_LSTM, val_loader)

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()



Osserviamo che il modello LSTM Semplice presenta un comportamento molto anomalo.

Dal classification report vediamo:
- Recall 1.00 su una sola classe: il modello sta predicendo tutte le classi come una unica classe
- Recall 0.00 sulle altre: il modello sta completamente ignorando le altre due
- Accuracy 0.33: il valore è probabilmente da attribuire alla distribuzione bilanciata del dataset (1/3 per classe).

La confusion matrix conferma che c’è un forte bias verso una sola classe.
Questo dimostra che LSTM è un modello troppo semplice che non riesce a generalizzare bene sul nostro dataset.

Potremmo provare ad utilizzare embedding pre-addestrati (GloVe) e aumentare il numero di epoche per permettere una convergenza migliore del modello, ma il problema principale rimane la monodirezionalità di una rete LSTM che non riesce a catturare il contesto della parola da destra a sinistra.

Quindi optiamo per una versione avanzata del modello che prevede bidirezionalità: BiLSTM.


# **BIDIRECTIONAL LSTM: BiLSTM**


Addestrato il modello LSTM, al fine di massimizzare la performance, proviamo a lavorare con una sua versione avanzata : il BiLSTM.
Attraverso l'impiego di due LSTM: il forward e il backward, ogni parola viene rappresentata da informazioni precedenti e successive alla parola stessa.
In un task di textual entitelment questo consente di avere una comprensione semantica e sintattica più ricca e completa.




Definiamo il modello neurale, riprendendo il codice relativo al modello LSTM semplice, attivando il parametro di bidirezionalità (bidirectional=True).

Ripetiamo dunque i processi di training e validation.

In [ ]:
class SentenceClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, output_dim=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 4, output_dim)

    def forward(self, premise, hypothesis):

        prem_emb = self.embedding(premise)
        hypo_emb = self.embedding(hypothesis)


        _, (prem_hidden, _) = self.lstm(prem_emb)
        _, (hypo_hidden, _) = self.lstm(hypo_emb)


        prem_rep = torch.cat([prem_hidden[0], prem_hidden[1]], dim=1)
        hypo_rep = torch.cat([hypo_hidden[0], hypo_hidden[1]], dim=1)


        combined = torch.cat([prem_rep, hypo_rep], dim=1)
        output = self.fc(combined)
        return output


**INIZIALIZZAZIONE:**

In [ ]:
class SNLIModel(pl.LightningModule):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, num_classes=3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 4, num_classes)

        self.loss_fn = nn.CrossEntropyLoss()
        self.lr = lr

    def forward(self, premise, hypothesis):
        prem_emb = self.embedding(premise)
        hypo_emb = self.embedding(hypothesis)

        _, (prem_hn, _) = self.lstm(prem_emb)
        _, (hypo_hn, _) = self.lstm(hypo_emb)

        prem_repr = torch.cat((prem_hn[0], prem_hn[1]), dim=-1)
        hypo_repr = torch.cat((hypo_hn[0], hypo_hn[1]), dim=-1)

        combined = torch.cat([prem_repr, hypo_repr], dim=-1)

        return self.classifier(combined)

    def training_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        preds = torch.argmax(logits, dim=1)
        acc = (preds == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


**DEFINIZIONE DEL TRAINER:**

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="bilstm_model"
)


model_BiLSTM = SNLIModel(vocab_size=len(vocab))


device = "cuda" if torch.cuda.is_available() else "cpu"
model_BiLSTM = model_BiLSTM.to(device)


trainer = Trainer(
    max_epochs=20,
    accelerator="gpu" if device == "cuda" else "cpu",
    devices=1,
    callbacks=[early_stop_callback],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=10
)


**ADDESTRAMENTO:**

In [ ]:
trainer.fit(model_BiLSTM, train_loader, val_loader)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/bilstm_model/

**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**

In [ ]:
def evaluate_model(model, dataloader):
    model = model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione in corso"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())


    return all_labels, all_preds


y_true, y_pred = evaluate_model(model_BiLSTM, val_loader)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()



Il modello BiLSTM semplice ha dimostrato performance significativamente migliori rispetto alla versione LSTM unidirezionale.

In particolare, l'accuratezza complessiva ha raggiunto il 65%.
Analizzando il classification report notiamo che:
- La classe entailment è riconosciuta con maggiore facilità, ottenendo precisione 0.66, recall 0.70 e F1-score 0.68.
- Le classi neutral e contradiction, spesso più ambigue e complesse da distinguere, hanno comunque raggiunto valori leggermente più bassi ma comunque ottimi, segnalando che il modello è in grado di catturare anche le sfumature logiche più complesse tra premessa e ipotesi.

La confusion matrix conferma la buona capacità del modello di generalizzare: le predizioni risultano distribuite in modo piuttosto bilanciato tra le tre classi.

Il BiLSTM costituisce quindi una solida baseline per il nostro task e rappresenta un’ottima base da cui partire per ulteriori ottimizzazioni, come l’introduzione di word embeddings pre-addestrati (GLoVe) o strutture di classificazione più profonde (MLP).



-------

OTTIMIZZAZIONE CON GLOVE

**DOWNLOAD DEGLI EMBEDDING PRE-ADDESTRATI (GLoVE):**

Gli embeddings sono rappresentazioni vettoriali dense di parole che codificano informazioni semantiche, invece di rappresentare le parole come one-hot vectors. In particolare impiegheremo un tipo di embedding pre-addestrato(matrice di embedding già pronta) denominato [GLoVE](https://nlp.stanford.edu/projects/glove/), utile a catturare relazioni semantiche nel testo.


Scarichiamo il file `glove.6B.50d.txt`, contenente embedding pre-addestrati da 50 dimensioni, da un mirror GitHub per evitare il download completo dell'archivio originale.

In [ ]:
!curl -L -o glove.6B.50d.txt "https://huggingface.co/JeremiahZ/glove/resolve/main/glove.6B.50d.txt"

Carichiamo i vettori GloVe filtrati da `glove.6B.50d.txt`, limitandoci solo alle parole presenti nel vocabolario (`vocab`) costruito dai dati SNLI. Questo riduce l'uso di memoria e velocizza il processo di embedding.

In [ ]:
glove_path = "glove.6B.50d.txt"
embedding_dim = 50
glove_vectors = {}




target_words = set(vocab.keys())

with open(glove_path, "r", encoding="utf8") as f:
    for line in tqdm(f, desc="Caricamento GloVe filtrato"):
        parts = line.strip().split()
        word = parts[0]
        if word in target_words:
            vector = list(map(float, parts[1:]))
            glove_vectors[word] = vector

print(f"Vettori GloVe trovati nel vocab: {len(glove_vectors)} / {len(vocab)}")


**CREAZIONE DELLA MATRICHE DI EMBEDDING:**

Costruiamo la matrice di embedding (`embedding_matrix`) a partire dai vettori GloVe filtrati. Per ogni parola nel vocabolario, inseriamo il vettore GloVe corrispondente; se assente, usiamo un vettore casuale normalizzato.

In [ ]:
embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, idx in tqdm(vocab.items(), desc="Creazione embedding matrix"):
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))  # fallback random

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)
print("Matrice di embedding pronta:", embedding_matrix.shape)


**DEFINIZIONE DEL MODELLO AVANZATO BiLSTM (con GloVe, Dropout, 2 layer, bidirezionale):**

In questa cella definiamo il modello `SNLIModel` come sottoclasse di `LightningModule`.
Il modello utilizza:
- embedding pre-addestrati (GloVe): impostiamo Freeze=False, per permettere il fine-tuning degli embeddings(addattamento di un modello già addestrato su un dataset specifico)
- 2 layer: il secondo layer rielabora il contesto prodotto dal primo, catturando strutture sintattiche e logiche più profonde nelle frasi.
- dropout: tecnica di regolarizzazione, che porta spegnimento casuale di alcuni neuroni durante il training, costringendo il modello ad imparare rappresentazioni più robuste, riducendo l’overfitting.

L’output dei due LSTM (premessa e ipotesi) viene concatenato e passato a un classificatore.

In [ ]:
class SNLIModel(pl.LightningModule):
    def __init__(self,
                 vocab_size,
                 embedding_dim=50,
                 hidden_dim=128,
                 num_classes=3,
                 dropout_p=0.3):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embeddings=embedding_matrix,
            freeze=False
        )

        self.dropout = nn.Dropout(dropout_p)


        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )


        self.classifier = nn.Linear(hidden_dim * 4, num_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, premise, hypothesis):

        prem_emb = self.dropout(self.embedding(premise))
        hypo_emb = self.dropout(self.embedding(hypothesis))


        _, (prem_h, _) = self.lstm(prem_emb)
        _, (hypo_h, _) = self.lstm(hypo_emb)


        prem_out = torch.cat((prem_h[-2], prem_h[-1]), dim=1)
        hypo_out = torch.cat((hypo_h[-2], hypo_h[-1]), dim=1)

        combined = torch.cat((prem_out, hypo_out), dim=1)
        logits = self.classifier(combined)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


**INIZIALIZZAZIONE:**

Inizializziamo il modello `SNLIModel` passando la dimensione del vocabolario e utilizzando i parametri precedentemente definiti.

In [ ]:
model_BiLSTMO = SNLIModel(vocab_size=len(vocab))


**DEFINIZIONE DEL TRAINER CON EARLYSTOPPING E GRADIENT CLIPPING:**

Nelle celle seguenti avviamo l’addestramento del modello `SNLIModel` utilizzando PyTorch Lightning.  
È attivata una callback di EarlyStopping basata sulla `val_loss` e il  [Gradient Clipping](https://lightning.ai/docs/pytorch/stable/common/trainer.html#gradient-clipping) per evitare esplosioni del gradiente. L’allenamento termina automaticamente se la validazione non migliora per 5 epoche consecutive.

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="bilstmo_model"
)

trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    gradient_clip_val=1.0,
    callbacks=[early_stop_callback],
    logger=logger,
    log_every_n_steps=10,
    enable_progress_bar=True
)




**ADDESTRAMENTO:**

In [ ]:
trainer.fit(model_BiLSTMO, train_loader, val_loader)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/bilstmo_model/

**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**


In questa sezione valutiamo il modello BiLSTM avanzato dopo l’addestramento con EarlyStopping. Calcoliamo le metriche di classificazione (precision, recall, f1-score) e visualizziamo la matrice di confusione.


In [ ]:
def evaluate_model(model, dataloader):
    model = model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione in corso"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return all_labels, all_preds


y_true, y_pred = evaluate_model(model_BiLSTMO, val_loader)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


Il modello BiLSTM ottimizzato ha ottenuto un notevole miglioramento rispetto alle versioni precedenti, portando l'accuratezza complessiva al 68%.
Il classification report ci informa che:
-	l’Entailment rimane la classe più facile da predire: ha un F1-score di 0.70, una buona precision (0.68) e un’ottima recall (0.71).
-	Neutral e Contradiction raggiungono entrambe un F1-score di 0.67, con miglioramenti sia nella precisione che nel recall rispetto al BiLSTM semplice.
La confusion matrix conferma la bontà delle performance. I casi di confusione tra neutral e contradiction sono ancora presenti, ma ridotti grazie all'arricchimento semantico introdotto dagli embedding GLoVe.


# **OTTIMIZZAZIONE BLSTM CON MLP**

Dopo aver ottenuto i risultati, non soddisfatti di questi, abbiamo provato ad ottimizzare ulteriormente: partendo dal modello precedente, sostituiamo il linear layer con un MLP (rete neurale feedforward con uno o più layer lineari "fully connected" e attivazioni non lineari).
In questo modo il modello può apprendere pattern non lineari e ridurre l' overfitting attraverso il dropout integrato.

Aggiorniamo quindi solo il codice del costruttore init.
Il resto del modello non cambia, perché il combined ha ancora dimensione hidden_dim * 4, quindi si collega perfettamente all’MLP.

In [ ]:
class SNLIModel(pl.LightningModule):
    def __init__(self,
                 vocab_size,
                 embedding_dim=50,
                 hidden_dim=128,
                 num_classes=3,
                 dropout_p=0.3):
        super().__init__()


        self.embedding = nn.Embedding.from_pretrained(
            embeddings=embedding_matrix,
            freeze=False
        )

        self.dropout = nn.Dropout(dropout_p)


        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )


        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )


        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, premise, hypothesis):

        prem_emb = self.dropout(self.embedding(premise))
        hypo_emb = self.dropout(self.embedding(hypothesis))


        _, (prem_h, _) = self.lstm(prem_emb)
        _, (hypo_h, _) = self.lstm(hypo_emb)


        prem_out = torch.cat((prem_h[-2], prem_h[-1]), dim=1)
        hypo_out = torch.cat((hypo_h[-2], hypo_h[-1]), dim=1)


        combined = torch.cat((prem_out, hypo_out), dim=1)


        logits = self.classifier(combined)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


**DEFINIZIONE DEL TRAINER:**

In [ ]:
model_BiLSTMMLP = SNLIModel(vocab_size=len(vocab))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training su:", device)


early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="bilstm_mlp_model"
)


trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    callbacks=[early_stop_callback],
    logger=logger,
    gradient_clip_val=1.0,
    enable_progress_bar=True,
    log_every_n_steps=10
)



**ADDESTRAMENTO:**

In [ ]:
trainer.fit(model_BiLSTMMLP, train_loader, val_loader)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/bilstm_mlp_model/


**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**

In [ ]:
def evaluate_model(model, dataloader):
    model = model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione in corso"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return all_labels, all_preds


y_true, y_pred = evaluate_model(model_BiLSTMMLP, val_loader)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

Il modello BiLSTM arricchito con un MLP finale mostra un ulteriore miglioramento rispetto alle versioni precedenti, raggiungendo una accuratezza dell’76%, la più alta tra i modelli testati fino ad ora nel progetto.

Il classification report ci mostra che:
-	Entailment ha un F1-score di 0.79 con recall molto alto (0.83) e precision di 0.76. Il modello rimane molto efficace nell'identificare relazioni di implicazione.
-	Neutral presenta un F1-score 0.72 leggermente più basso, ma comunque stabile e superiore ai modelli precedenti. Rimane la classe più difficile da distinguere.
-	Contradiction infine, con un F1-score di 0.78, presenta la precision più alta tra tutte le classi, raggiungendo un valore di 0.81, il che suggerisce che quando il modello predice contraddizione, è molto sicuro.

La Confusion Matrix ci mostra che i falsi positivi tra entailment-neutral e neutral-contradiction sono ancora presenti ma significativamente ridotti.
È evidente che il MLP finale ha migliorato la capacità discriminativa del modello, permettendo di apprendere relazioni più complesse tra le rappresentazioni finali di premise e hypothesis.



----


CONVOLUTIONAL NETWORK (CNN SEMPLICE)

Dopo aver esplorato modelli più sequenziali come LSTM e BiLSTM, abbiamo deciso di sperimentare anche con una CNN, reti  utilizzate nel campo del Natural Language Processing per diversi compiti di classificazione, grazie alla loro capacità di catturare pattern locali come n-grammi, ridurre il numero di parametri rispetto a modelli ricorrenti, rendendole più veloci da addestrare e generalizzare bene su grandi quantità di testo, anche senza necessariamente processare l'intera sequenza in ordine.

Infatti la struttura convoluzionale 1D è in grado di identificare pattern informativi sia nella premessa che nell'ipotesi, anche quando si trovano in posizioni diverse.

**DEFINIZIONE DEL MODELLO:**

Definiamo la classe del modello SNLI_CNN:
- L'embedding layer trasforma ogni parola (indice) in un vettore denso di embedding_dim dimensioni e padding_idx evita che il token <PAD> influenzi il modello, quindi verrà ignorato nei gradienti.
- Creiamo più layer convolutivi, definendo la dimensione dell’embedding (quanti canali "in ingresso"), quanti filtri applicare in output e quanto “ampio” è il filtro (kernel_size).
- Definiamo la funzione ausiliaria conv_and_pool che fa convoluzione (per trovare pattern nelle frasi), applica la ReLu per salvare solo i pattern positivi, tra questi estrae il pattern più forte (con max pooling) e infine fa uno sqeeze a due dimensioni (siccome rimane una signola attivazione forte per filtro, rimuove la dimensione temporale).
- Nel passo forward facciamo la permute che serve a trasformare l’output dell’Embedding in un input compatibile con Conv1d, riordinando le dimensioni: passiamo da batch, length, dim, a 	batch, channels, length (dove channels in questo caso corrisponde alla dimensione dell'embedding).
- Applichiamo quindi ciascun filtro a premessa e ipotesi, ottenendo un vettore per ciascun kernel.
- Concateniamo tutti i vettori ottenuti da ciascun kernel per la premessa e per l’ipotesi. Applichiamo dropout per regolarizzare. Questo risultato è pronto per passare al Linear layer per la classificazione finale in 3 classi.
- Definiamo il training step e il validation step
- Usiamo come ottimizzatore Adam, che è lo standard per modelli NLP di questo tipo.




In [ ]:
class SNLI_CNN(pl.LightningModule):
    def __init__(self, vocab_size, embedding_dim=50, num_filters=100, output_dim=3, kernel_sizes=[3,4,5], dropout=0.3, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim,
                      out_channels=num_filters,
                      kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes) * 2, output_dim)
        self.loss_fn = nn.CrossEntropyLoss()

    def conv_and_pool(self, x, conv):
        x = torch.relu(conv(x))
        x = torch.max_pool1d(x, x.shape[2]).squeeze(2)
        return x

    def forward(self, premise, hypothesis):
        prem_emb = self.embedding(premise).permute(0, 2, 1)
        hypo_emb = self.embedding(hypothesis).permute(0, 2, 1)

        prem_features = [self.conv_and_pool(prem_emb, conv) for conv in self.convs]
        hypo_features = [self.conv_and_pool(hypo_emb, conv) for conv in self.convs]

        combined = torch.cat(prem_features + hypo_features, dim=1)
        combined = self.dropout(combined)

        return self.fc(combined)

    def training_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self.forward(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


**INIZIALIZZAZIONE:**

Siccome abbiamo già embedding_matrix, vocab e dataset, possiamo iniziare direttamente con la definizione del modello SNLI_CNN:
- Passiamo al modello la dimensione del vocabolario (numero di parole uniche inclusi <PAD> e <UNK>).
- Impostiamo la larghezza dei vettori di embedding che deve coincidere con la dimensione di GloVe utilizzata (glove.6B.50d.txt)
- Scegliamo num_filters=100 per ciascun kernel: un valore di default ampiamente usato nella letteratura NLP con CNN , in quanto abbastanza potente da catturare pattern significativi rimanendo leggero e veloce da addestrare.
- Il numero di output è impostato a 3 (3 labels)
- Scegliamo 3-grammi, 4-grammi e 5-grammi come dimensioni dei filtri convoluzionali che scorrono sul testo.
- Impostiamo dropout=0.3 (spegnerà il 30% dei neuroni in modo casuale per evitare overfitting).

Source: [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/abs/1408.5882).








In [ ]:
model_CNN = SNLI_CNN(
    vocab_size=len(vocab),
    embedding_dim=50,
    num_filters=100,
    output_dim=3,
    kernel_sizes=[3, 4, 5],
    dropout=0.3
)


**DEFINIZIONE DEL TRAINER:**

Nel definire il trainer impostiamo un numero massimo di 20 epoche (solitamente per compiti CNN su SNLI spesso si allenano per 10–30 epoche), usiamo l'EarlyStopping (con patience 5), e logghiamo i valori ogni 10 batch (per una visione grafica più pulita).


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="cnn_model"
)


progress_bar = TQDMProgressBar(refresh_rate=10)

trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    callbacks=[early_stop, progress_bar],
    logger=logger,
    log_every_n_steps=10
)


**ADDESTRAMENTO**

In [ ]:
trainer.fit(model_CNN, train_loader, val_loader)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/cnn_model/


**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**

In [ ]:
def evaluate_model(model, dataloader):
    model.eval()
    model.to(device)
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return all_labels, all_preds



y_true, y_pred = evaluate_model(model_CNN, val_loader)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


Il modello CNN fornisce performance solide e bilanciate tra le classi, con accuratezza del 66%. Tuttavia, manca la profondità nella comprensione sequenziale e contestuale, il che spiega i risultati leggermente inferiori rispetto al modello BiLSTM+MLP.

Il classification report ci illustra che:
-	Gli errori sono principalmente tra le classi "neutral" e "contradiction", che sono semanticamente più simili e quindi più facilmente confuse.
-	La precisione è molto simile per tutte le classi, segno di buona generalizzazione

La cosa più interessante però è che il modello distribuisce bene le predizioni tra le 3 classi, evitando il classico bias verso "entailment". Questo accade perché:
-	 La CNN non elabora le sequenze parola per parola (come fanno gli LSTM), ma analizza in parallelo finestrature locali (n-grammi). Questo approccio permette al modello di non essere influenzato dalla posizione delle parole, quindi non rischia di imparare pattern sequenziali sbilanciati verso classi frequenti.
-	La CNN cattura combinazioni di parole forti e pattern semantici locali (es. “not the same”, “is equal to”). Questo è molto utile per distinguere contraddizione vs entailment, dove la differenza è spesso legata a piccoli dettagli.


## **OTTIMIZZAZIONE: CNN + MLP**

Il modello CNN semplice  estrae feature (pattern locali) dalle frasi, concatena i vettori (premessa + ipotesi) e li manda direttamente in un classificatore lineare.
Per aumentare la capacità del modello di rappresentare interazioni tra pattern, possiamo provare a introdurre una seconda non linearità sulla rappresentazione globale della frase: sostituiamo l'ultimo strato lineare della CNN con una rete MLP (Multi-Layer Perceptron), in modo che sia facilitata la discriminazione tra classi complesse.



**DEFINIZIONE:**

Nella definizione della CNN integrata con MLP, restano invariati le seguenti componenti: l'embedding layer, le convoluzioni 1D multiple con kernel diversi, pooling per ogni convoluzione e la concatenazione dei vettori.

L'unica differenza consiste nella definizione del classifier: al posto di nn.linear (che classifica direttamente senza passare da uno strato intermedio) usiamo nn.Sequential (che aggiunge una rete neurale tra l’output delle CNN e le classi).



In [ ]:
class CNNWithMLP(pl.LightningModule):
    def __init__(self,
                 vocab_size,
                 embedding_dim=50,
                 num_filters=100,
                 kernel_sizes=[3, 4, 5],
                 hidden_dim=256,
                 num_classes=3,
                 dropout=0.3,
                 embedding_matrix=None,
                 lr=1e-3):
        super().__init__()
        self.save_hyperparameters()


        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.dropout = nn.Dropout(dropout)


        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim,
                      out_channels=num_filters,
                      kernel_size=k)
            for k in kernel_sizes
        ])


        self.classifier = nn.Sequential(
            nn.Linear(num_filters * len(kernel_sizes) * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

        self.loss_fn = nn.CrossEntropyLoss()

    def conv_and_pool(self, x, conv):
        x = conv(x)
        x = torch.relu(x)
        x = torch.max_pool1d(x, kernel_size=x.shape[2])
        return x.squeeze(2)

    def forward(self, premise, hypothesis):
        prem_emb = self.embedding(premise).permute(0, 2, 1)
        hypo_emb = self.embedding(hypothesis).permute(0, 2, 1)

        prem_features = [self.conv_and_pool(prem_emb, conv) for conv in self.convs]
        hypo_features = [self.conv_and_pool(hypo_emb, conv) for conv in self.convs]

        combined = torch.cat(prem_features + hypo_features, dim=1)
        return self.classifier(combined)

    def training_step(self, batch, batch_idx):
        logits = self(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["premise"], batch["hypothesis"])
        loss = self.loss_fn(logits, batch["label"])
        acc = (logits.argmax(dim=1) == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


**INIZIALIZZAZIONE:**

In [ ]:
model_CNNMLP = CNNWithMLP(
    vocab_size=len(vocab),
    embedding_dim=50,
    num_filters=100,
    kernel_sizes=[3, 4, 5],
    hidden_dim=256,
    num_classes=3,
    dropout=0.3,
    embedding_matrix=embedding_matrix,
    lr=1e-3
)


**DEFINIZIONE DEL TRAINER:**

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True
)


progress_bar = TQDMProgressBar(refresh_rate=10)


logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="cnn_mlp_model"
)


trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    callbacks=[early_stop, progress_bar],
    logger=logger,
    log_every_n_steps=10
)

**ADDESTRAMENTO:**

In [ ]:
trainer.fit(model_CNNMLP, train_loader, val_loader)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/cnn_mlp_model/


**VALUTAZIONE DELLE PRESTAZIONI DEL MODELLO:**

In [ ]:
def evaluate_model(model, dataloader):
    model.eval()
    model.to(device)
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Valutazione"):
            premise = batch["premise"].to(device)
            hypothesis = batch["hypothesis"].to(device)
            labels = batch["label"].to(device)

            outputs = model(premise, hypothesis)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return all_labels, all_preds


y_true, y_pred = evaluate_model(model_CNNMLP, val_loader)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["entailment", "neutral", "contradiction"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entailment", "neutral", "contradiction"],
            yticklabels=["entailment", "neutral", "contradiction"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


Il modello CNN+MLP mostra prestazioni complessivamente solide, con un accuracy del 71% e metriche ben bilanciate.

L'aggiunta dell'MLP dopo il blocco convoluzionale ha migliorato la capacità del modello di catturare pattern non lineari e relazioni complesse, traducendosi in un miglioramento delle performance rispetto alla CNN semplice.

La confusion matrix mostra una buona distinzione tra le tre classi, con errori contenuti. Notiamo che la classe neutral rimane leggermente più problematica da distinguere, probabilmente per la natura ambigua di alcune frasi.


CONCLUSIONE COMPARATIVA

Nella seguente tabella riassumiamo i risultati ottenuti, evidenziando per ciascun modello l’accuratezza, i punti di forza principali e i limiti riscontrati durante la sperimentazione.

Dall’analisi comparativa, il modello BiLSTM + GloVe + MLP è risultato essere il più bilanciato in termini di performance, generalizzazione e comprensione semantica.
L’integrazione di GloVe ha permesso di arricchire la rappresentazione delle frasi con informazioni semantiche pre-addestrate, mentre la rete MLP ha migliorato la capacità di classificazione, gestendo anche relazioni più sottili tra le frasi.

L'accuracy ottenuta è del 77%: questo ci indica che, seppur lavorando su un dataset complesso (che contiene frasi ambigue, con sfumature semantiche, sinonimi, implicazioni logiche e contraddizioni sottili) e senza l'utilizzo di modelli pre-addestrati, il modello riesce comunque a cogliere relazioni semantiche complesse senza overfitting rappresentando una soluzione realistica e ben ottimizzata.






In [ ]:
import pandas as pd

# Mostra tutto il contenuto delle celle
pd.set_option('display.max_colwidth', None)

# Tabella dei risultati
df_comparativa = pd.DataFrame({
    "Modello": [
        "Logistic Regression",
        "Logistic Regression + TF-IDF",
        "LSTM semplice",
        "BiLSTM",
        "BiLSTM + GloVe",
        "BiLSTM + MLP",
        "CNN semplice",
        "CNN + MLP"
    ],
    "Accuracy": [0.59, 0.59, 0.33, 0.65, 0.68, 0.76, 0.66, 0.71],
    "Punti di forza": [
        "Modello semplice e interpretabile",
        "Migliora la rappresentazione testuale con pesi TF-IDF",
        "Cattura le dipendenze sequenziali tra parole",
        "Considera contesto da sinistra a destra e viceversa",
        "Integra semantica grazie agli embedding GloVe",
        "Introduce non-linearità migliorando la classificazione",
        "Cattura efficacemente pattern locali nelle frasi",
        "Combina pattern locali con classificazione più sofisticata"
    ],
    "Limiti": [
        "Non cattura relazioni complesse",
        "Nessuna informazione sul contesto sequenziale",
        "Grave overfitting: predice solo una classe",
        "Prestazioni ancora migliorabili tra classi ambigue",
        "Confusione tra neutral e contradiction",
        "Maggiore complessità e tempi di addestramento",
        "Non adatto per sequenze lunghe",
        "Meno performante del BiLSTM+MLP"
    ]
})

# Visualizza la tabella
df_comparativa



RIFLESSIONE CONCLUSIVA

Questo progetto nasce dall’incontro tra due mondi: la Filosofia del linguaggio e l’Intelligenza Artificiale.

Nel corso del lavoro abbiamo affrontato il problema della Natural Language Inference (NLI): un compito che richiede ad una macchina di comprendere relazioni semantiche tra frasi, distinguendo tra entailment, neutralità e contraddizione.

Ma dietro questa computazione, si nasconde una questione profondamente filosofica: *una macchina può davvero “capire” un enunciato, oppure si limita a riconoscere correlazioni statistiche tra parole?*

Attraverso l’uso dei modelli Logistic Regression, LSTM, BiLSTM, CNN, MLP e l’integrazione di embedding semantici come GloVe, abbiamo cercato di costruire architetture in grado di catturare il significato implicito tra due frasi.
Ma ogni modello, per quanto preciso, non ha superato il test:
*l’AI non interpreta, ma ricalca pattern appresi; non coglie il senso, ma lo simula.*
